# Colab run v4: corrected source data, category-level clothing evaluation, retrained adapter
Changes vs v3 (all in `src/`): ASTE text is **detokenised**, duplicate (aspect, sentiment) gold entries are **merged**, dev/test contain **aspect-free sentences**, relaxed matching is **token-level**, and the clothing evaluation is **category-level** (gold labels are mostly categories, not spans; both sides map to `configs/canonical_aspects.json`; span-level literal-gold score and net-sentiment agreement are reported too; text comes from the pipeline data). Because training data changed, the fine-tuned adapter is **retrained** (new path `absa-qlora-v4`). All old results are invalid and archived.

Upload `capstone-absa-colab-v4.zip` to `/content`, set **Runtime > T4 GPU**, run in order. Training takes roughly an hour; it resumes from Drive checkpoints if the runtime disconnects.

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
ZIP, PROJECT = "/content/capstone-absa-colab-v4.zip", "/content/capstone-absa-colab"
if not os.path.exists(ZIP):
    from google.colab import files; files.upload()
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(PROJECT)
%cd /content/capstone-absa-colab
!pip install -q -r requirements.txt pytest
!pip freeze > pip_freeze.txt
!python -m spacy download en_core_web_sm -q
MODEL = "unsloth/Llama-3.2-3B-Instruct"
ADAPTER = "/content/drive/MyDrive/absa-qlora-v4"   # NEW adapter, trained on the corrected data

In [ ]:
# Rebuild data with the v4 preprocessing, then unit + data-invariant tests
import os
if not os.path.exists("data/raw/amazon_clothing_sample.jsonl"):
    !python scripts/fetch_data.py amazon --n 5000 --out data/raw/amazon_clothing_sample.jsonl
!python -m src.prepare_data && python -m src.build_sft_data && python -m src.prepare_clothing
!python -m pytest -q tests

In [ ]:
# Archive any OLD (v3) results from Drive, then start clean
import shutil, os, time
old = f"/content/drive/MyDrive/absa-results-old-{int(time.time())}"
if os.path.isdir("/content/drive/MyDrive/absa-results"): shutil.copytree("/content/drive/MyDrive/absa-results", old); print("old results archived to", old)
for f in ("baseline_dev","baseline_test","prompted_dev","prompted_test","finetuned_dev","finetuned_test","clothing_baseline","clothing_baseline_lr","clothing_prompted","clothing_finetuned"):
    p=f"results/{f}.json"
    if os.path.exists(p): os.remove(p)

In [ ]:
# Baselines on the corrected source dev/test (these are the in-domain reference for the drop)
!python -m src.baseline_spacy --split dev
!python -m src.baseline_spacy --split test

In [ ]:
# QLoRA fine-tuning on the corrected data (best eval-loss epoch is kept; resumes from Drive checkpoints)
# If you hit CUDA out-of-memory, rerun with --batch 2
!python -m src.finetune_qlora --model $MODEL --out $ADAPTER --epochs 3 --batch 4
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "training did not finish

In [ ]:
# DIAGNOSTIC (2 min): raw outputs of the NEW adapter on real clothing sentences (v3 returned [] for ~all of them)
import sys; sys.path.insert(0, ".")
from src.common import read_jsonl
from src.llm_prompted import hf_generator, extract_with_retry
rows = read_jsonl("data/processed/clothing_reviews.jsonl")[:12]
sents = [s for r in rows for s in r["sentences"]][:40]
gen = hf_generator(MODEL, ADAPTER)
empty = 0
for s in sents:
    r = extract_with_retry(s, gen, few_shot=False)
    empty += (r["ok"] and not r["items"])
    print(repr(s[:110]), "->", r["raw"].strip()[:140])
print(f"\nempty predictions: {empty}/{len(sents)}")
del gen

In [ ]:
# Prompted LLM, source dev + test
for split in ("dev", "test"):
    !python -m src.llm_prompted --split $split --model $MODEL

In [ ]:
# Fine-tuned LLM, source dev + test
for split in ("dev", "test"):
    !python -m src.llm_prompted --split $split --model $MODEL --adapter $ADAPTER

In [ ]:
# Clothing evaluation, all methods, then comparison
for m in ("baseline", "baseline_lr"):
    !python -m src.eval_clothing --method $m --model $MODEL
!python -m src.eval_clothing --method prompted --model $MODEL
!python -m src.eval_clothing --method finetuned --model $MODEL --adapter $ADAPTER
!python -m src.compare

In [ ]:
# Aggregation for the report view with the LLM extractors
!python -m src.aggregate --method finetuned --adapter $ADAPTER --model $MODEL --limit 300 --min_mentions 5
!python -m src.aggregate --method prompted --model $MODEL --limit 300 --min_mentions 5

In [ ]:
# Save everything
import shutil
shutil.copy("pip_freeze.txt", "results/pip_freeze.txt")
shutil.copytree("results", "/content/drive/MyDrive/absa-results-v4", dirs_exist_ok=True)
shutil.make_archive("/content/capstone-absa-colab-v4-results", "zip", "results")
from google.colab import files; files.download("/content/capstone-absa-colab-v4-results.zip")
print(open("results/comparison.md").read())

## Only if the diagnostic still shows the fine-tuned model returning `[]` for most clothing sentences
Retrain with fewer epochs: `!python -m src.finetune_qlora --model $MODEL --out $ADAPTER-v2 --epochs 2 --batch 4`, point `ADAPTER` at it and rerun the fine-tuned cells.